# Entrenamiento del modelo de visión — Tiendita IA

**Tarea:** clasificar la foto de un producto (uno a la vez, como en el cobro). **Modelo base:** YOLO11n-cls (ImageNet). **Herramienta NVIDIA:** TensorRT.

Corre en el entorno GPU del **NVIDIA DLI** (Tesla T4). **Guarda este notebook con sus salidas**: es parte del registro de ejecución (entregable C).

⏱ **El tiempo de GPU es limitado.** Las fotos se extraen de los videos en la laptop (`preparar_videos.py`); aquí solo se entrena.

**Antes de empezar**, sube con el panel de archivos de Jupyter:
1. `vision.zip` — la carpeta `vision/` del repo (sin `raw/`, `videos/` ni `dataset/`).
2. `raw.zip` — las fotos: `raw/<sesion>/<clase>/*.jpg`.

## 0. Diagnóstico

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!python -c "import urllib.request; urllib.request.urlopen('https://pypi.org', timeout=5); print('INTERNET OK')" || echo "SIN INTERNET: avisar para subir wheels a mano"
!df -h . | tail -1

## 1. Descomprimir código y fotos

In [ ]:
!unzip -q -o vision.zip -d . && unzip -q -o raw.zip -d vision/
!ls vision/raw && ls vision/raw/*/ | head -30

## 2. Instalación (~3–5 min)
PyTorch desde el índice **cu121**: compatible con el driver 545 de la T4.

In [ ]:
%pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r vision/requirements.txt

In [ ]:
%cd vision
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

## 3. División train / val / test por sesión
Ninguna sesión puede estar en dos conjuntos. Ajusta si sus sesiones se llaman distinto.

In [ ]:
!python split.py --train s1 --test s2

## 4. Prueba de humo (2 épocas, ~1 min)
Verifica que todo corre antes del entrenamiento completo.

In [ ]:
!python train.py --nombre humo --epochs 2

## 5. Entrenamiento completo (~5–15 min en T4)

In [ ]:
!python train.py --nombre v1 --epochs 30

## 6. Evaluación con el conjunto de prueba

In [ ]:
!python eval.py --nombre v1

In [ ]:
from IPython.display import Image, display
display(Image("artefactos/v1/eval/val_test/confusion_matrix_normalized.png", width=650))

## 7. Optimización con TensorRT + benchmark

In [ ]:
!python export_trt.py --nombre v1

## 8. Empaquetar y DESCARGAR
Descarga `artefactos_v1.zip` (y este notebook guardado) **antes de cerrar el entorno**: los archivos pueden perderse.

In [ ]:
!rm -rf artefactos/humo
!zip -q -r ../artefactos_v1.zip artefactos/v1 dataset/estadisticas.json
!ls -lh ../artefactos_v1.zip && head -40 artefactos/v1/registro.json